# fft_approximate: approximate vs. exact range-aggregate queries

Compares `fft_approximate`'s `approximate_avg(...)` (computed over a truncated FFT coefficient representation) against an exact `AVG()` on the same raw rows, over the same time window. See this repo's README for what "approximate" means here and why `create_approximation(...)` isn't used (it's not implemented in the extension itself — this notebook computes coefficients in Python instead).

In [ ]:
import os
import time

import numpy as np
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host=os.environ.get("PGHOST", "127.0.0.1"),
    port=os.environ.get("PGPORT", "5432"),
    user=os.environ.get("PGUSER", "postgres"),
    password=os.environ.get("PGPASSWORD", ""),
    dbname=os.environ.get("PGDATABASE", "w4h"),
)
cur = conn.cursor()

schema = os.environ.get("W4H_DATASET_SCHEMA", "ds_sample_fitbit_csv")
table = os.environ.get("W4H_SIGNAL_TABLE", "fitbit_heart_rate")
subject_id = os.environ.get("W4H_SUBJECT_ID", "").strip()
if not subject_id:
    cur.execute(f"SELECT id FROM {schema}.{table} GROUP BY id ORDER BY count(*) DESC LIMIT 1")
    subject_id = cur.fetchone()[0]
print("subject:", subject_id)

## 1. Pull a bounded slice (not the full file)

This demo only needs a few thousand rows to make the point — never load a full multi-hundred-thousand-row signal just to run this notebook.

In [ ]:
N = 2048  # bounded slice size
cur.execute(
    f"SELECT value FROM {schema}.{table} WHERE id = %s ORDER BY date, timestamp LIMIT %s",
    (subject_id, N),
)
values = np.array([r[0] for r in cur.fetchall()], dtype=float)
N = len(values)
print(f"rows pulled: {N}")

## 2. Exact ground truth over a sub-window

In [ ]:
t1, t2 = N // 4, (3 * N) // 4  # a sub-window, not the whole slice
exact_mean_window = float(np.mean(values[t1:t2]))
print(f"exact mean over window [{t1}:{t2}]: {exact_mean_window:.3f}")

## 3. Compute FFT coefficients and load them

`create_approximation(...)` is not implemented in `fft_approximate` itself (confirmed in its README) — this cell does the equivalent work in Python with `numpy.fft.fft`, then inserts each coefficient as a `complex` literal (`'(re+imj)'::complex`).

In [ ]:
coeffs = np.fft.fft(values)  # length N, complex

cur.execute("DROP TABLE IF EXISTS demo_coeffs")
cur.execute("CREATE TABLE demo_coeffs (k integer PRIMARY KEY, c complex)")
rows = [(k, f"({c.real}{c.imag:+}j)") for k, c in enumerate(coeffs)]
cur.executemany("INSERT INTO demo_coeffs (k, c) VALUES (%s, %s::complex)", rows)
conn.commit()
print(f"coefficients inserted: {len(rows)}")

## 4. Approximate vs. exact, at increasing coefficient counts

Using only the first `K` (lowest-frequency) coefficients out of `N` — this is the actual compression/accuracy tradeoff `fft_approximate` is for. `K=N` (all coefficients) should recover the exact answer up to floating-point round-trip error through the `complex` type's text representation.

In [ ]:
print(f"{'K':>6} {'approx':>10} {'exact':>10} {'error %':>9} {'time (ms)':>10}")
for K in [1, 5, 20, 100, N]:
    t0 = time.perf_counter()
    cur.execute(
        "SELECT re(approximate_avg(c, %s, %s, %s ORDER BY k)) FROM demo_coeffs WHERE k < %s",
        (t1, t2, N, K),
    )
    approx = cur.fetchone()[0]
    dt_ms = (time.perf_counter() - t0) * 1000
    err_pct = abs(approx - exact_mean_window) / abs(exact_mean_window) * 100
    print(f"{K:6d} {approx:10.3f} {exact_mean_window:10.3f} {err_pct:9.3f} {dt_ms:10.2f}")

## 5. Exact query on raw rows, for timing comparison

At this bounded, small scale (a few thousand rows) both paths are sub-millisecond — the meaningful comparison here is accuracy vs. coefficient count (above), not timing. The extension's real timing advantage is at a scale this demo deliberately does not exercise: querying a compact coefficient table instead of scanning a much larger raw table (e.g. the bundled Fitbit sample's `calories.csv` is 1.2M+ rows).

In [ ]:
t0 = time.perf_counter()
cur.execute(
    f"SELECT AVG(value) FROM ("
    f"  SELECT value FROM {schema}.{table}"
    f"  WHERE id = %s ORDER BY date, timestamp OFFSET %s LIMIT %s"
    f") w",
    (subject_id, t1, t2 - t1),
)
exact_db = float(cur.fetchone()[0])
dt_ms = (time.perf_counter() - t0) * 1000
print(f"exact SQL AVG(): {exact_db:.3f}  time={dt_ms:.2f}ms")

In [ ]:
# Clean up the demo table — do not leave it behind
cur.execute("DROP TABLE IF EXISTS demo_coeffs")
conn.commit()
cur.close()
conn.close()